#Incremental Data loading

In [0]:
CREATE DATABASE Sales_scd;

In [0]:

CREATE TABLE Sales_scd.Orders
(
    OrderID INT,
    OrderDate DATE,
    CustomerID INT,
    CustomerName VARCHAR(100),
    CustomerEmail VARCHAR(100),
    ProductID INT,
    ProductName VARCHAR(100),
    ProductCategory VARCHAR(50),
    RegionID INT,
    Country VARCHAR(50),
    Quantity INT,
    UnitPrice DECIMAL(10,2),
    TotalAmount DECIMAL(10,2)
)

In [0]:
INSERT INTO Sales_scd.Orders
VALUES
(1021,'2024-02-21',21,'Ethan Walker','ethan.walker21@email.com',21,'Gaming Mouse','Accessories',1,'USA',2,45.00,90.00),
(1022,'2024-02-22',22,'Ava Hall','ava.hall22@email.com',22,'Mechanical Keyboard','Accessories',2,'Canada',1,120.00,120.00),
(1023,'2024-02-23',23,'Noah Allen','noah.allen23@email.com',23,'Smart Watch','Electronics',3,'UK',2,250.00,500.00),
(1024,'2024-02-24',24,'Ella Young','ella.young24@email.com',24,'Bluetooth Speaker','Electronics',4,'Germany',3,80.00,240.00),
(1025,'2024-02-25',25,'Liam Hernandez','liam.hernandez25@email.com',25,'Office Chair','Furniture',5,'France',1,350.00,350.00),
(1026,'2024-02-26',26,'Grace King','grace.king26@email.com',26,'Standing Desk','Furniture',1,'USA',1,500.00,500.00),
(1027,'2024-02-27',27,'Mason Wright','mason.wright27@email.com',27,'USB Hub','Accessories',2,'Canada',4,25.00,100.00),
(1028,'2024-02-28',28,'Chloe Scott','chloe.scott28@email.com',28,'Webcam HD','Accessories',3,'UK',2,95.00,190.00),
(1029,'2024-02-29',29,'Logan Green','logan.green29@email.com',29,'27 Inch Monitor','Electronics',4,'Germany',1,320.00,320.00),
(1030,'2024-03-01',30,'Zoe Adams','zoe.adams30@email.com',30,'External Hard Drive','Electronics',5,'France',2,140.00,280.00);

In [0]:
SELECT * FROM Sales_scd.orders

#Data Warehousing

In [0]:
CREATE DATABASE orderDWH;

In [0]:
-- intial Load
CREATE OR REPLACE TABLE orderDWH.stg_sales
AS 
SELECT * FROM Sales.orders

### Transformation


In [0]:
CREATE VIEW orderDWH.trans_sales
AS
SELECT * FROM orderDWH.stg_sales WHERE Quantity IS NOT NULL

In [0]:
select * from orderDWH.trans_sales

### Core Layer

#### DimCustomers

In [0]:
CREATE OR REPLACE TABLE orderDWH.DimCustomer
(
    CustomerID int,
    CustomerName string,
    CustomerEmail string,
    DimCustomerKey int
)

In [0]:
CREATE OR REPLACE VIEW orderDWH.View_DimCustomer
 AS
select Distinct_Customer.* , row_number() over(order by CustomerID) as DimCustomerKey
from
(select  
  distinct (CustomerID) as CustomerID ,
  CustomerName,
  CustomerEmail
from orderDWH.trans_sales
) AS Distinct_Customer


/* select  
  distinct (CustomerID) as CustomerID ,
  row_number() over(order by CustomerID) as DimCustomerKey,
  CustomerName,
  CustomerEmail
from salesDWH.trans_sales
*/

In [0]:
select * from orderDWH.View_DimCustomer

In [0]:
insert into orderDWH.DimCustomer
select * from orderDWH.View_DimCustomer

In [0]:
select * from orderDWH.DimCustomer

#### DimProducts

In [0]:
CREATE OR REPLACE TABLE orderDWH.DimProduct
(
    ProductID int,
    ProductName string,
    ProductCategory string,
    DimProductKey int
)

In [0]:
CREATE OR REPLACE VIEW orderDWH.View_DimProduct
 AS
SELECT  Distinct_Products.* , ROW_NUMBER()OVER (ORDER BY ProductID)
FROM
(
    SELECT 
      DISTINCT ProductID, 
      ProductName,
      ProductCategory
    FROM orderDWH.trans_sales
) AS Distinct_Products


In [0]:
insert into orderDWH.DimProduct 
select * from orderDWH.View_DimProduct

In [0]:
select * from orderDWH.DimProduct 

#### DimRegion

In [0]:
CREATE OR REPLACE TABLE orderDWH.DimRegion
(
    RegionID int,
    Country string,
    DimRegionKey int
)

In [0]:
CREATE OR REPLACE VIEW orderDWH.View_DimRegion
 AS
SELECT  Distinct_Region.* , ROW_NUMBER()OVER (ORDER BY RegionID)
FROM
(
    SELECT 
      DISTINCT RegionID as RegionID
    FROM orderDWH.trans_sales
) AS Distinct_Region

In [0]:
insert into orderDWH.DimRegion 
select * from orderDWH.View_DimRegion

In [0]:
select * from orderDWH.DimRegion 

####DimDate

In [0]:
CREATE OR REPLACE TABLE orderDWH.DimDate
(
    OrderDate DATE,
    DimDateKey int
)

In [0]:
CREATE OR REPLACE VIEW orderDWH.View_DimDate
 AS
SELECT  Distinct_Date.* , ROW_NUMBER()OVER (ORDER BY Distinct_Date.orderDate) as DimDateKey
FROM
(
    SELECT 
      DISTINCT (orderDate) as orderDate
    FROM orderDWH.trans_sales
) AS Distinct_Date

In [0]:
insert into orderDWH.DimDate 
select * from orderDWH.View_DimDate

In [0]:
select * from orderDWH.DimDate

### Fact Table


In [0]:
CREATE  OR REPLACE TABLE orderDWH.FactSales
(
    OrderID int,
    Quantity DECIMAL,
    UnitPrice DECIMAL,
    TotalAmount DECIMAL,
    DimProductKey int,
    DimCustomerKey int,
    DimRegionKey int,
    DimDateKey int
)



In [0]:
SELECT 
    F.OrderID,
    F.Quantity,
    F.UnitPrice,
    F.TotalAmount,
    DC.DimCustomerKey,
    DP.DimProductKey,
    DD.DimDateKey,
    DR.DimRegionKey
FROM 
    orderDWH.trans_sales F
LEFT JOIN orderDWH.DimCustomer DC
    ON F.CustomerID = DC.CustomerID
 LEFT JOIN orderDWH.Dimproduct DP
    ON F.ProductID = DP.ProductID
 LEFT JOIN orderDWH.DimDate DD
    ON F.OrderDate = DD.OrderDate
LEFT JOIN orderDWH.DimRegion DR
    ON F.RegionID = DR.RegionID




##  ========================================

## SCD TYPE `1`

In [0]:
CREATE OR REPLACE TABLE sales_scd.DimProduct
(
    ProductID int,
    ProductName string,
    ProductCategory string
)

In [0]:
CREATE OR REPLACE VIEW sales_scd.view_DimProduct
AS
SELECT DISTINCT(ProductID), ProductName, ProductCategory 
FROM sales_scd.orders
WHERE OrderDate > "2024-03-01"



In [0]:
select * from sales_scd.view_DimProduct

In [0]:
select * from sales_scd.DimProduct

In [0]:
-- initial Load
INSERT INTO sales_scd.DimProduct
SELECT ProductID, ProductName, ProductCategory 
FROM sales_scd.view_DimProduct

In [0]:
INSERT INTO Sales_scd.Orders
VALUES
(1051,'2024-03-03',29,'Logan Green','logan.green29@email.com',29,'HP Monitor','Electronics',4,'Germany',1,320.00,320.00),
(1052,'2024-03-04',30,'Zoe Adams','zoe.adams30@email.com',40,'Hard Drive Terabyte','Storage',5,'France',2,140.00,280.00)
;

In [0]:
select * from sales_scd.DimProduct

## MERG SCD TYPE `1`

In [0]:
MERGE INTO sales_scd.DimProduct trg
USING sales_scd.view_DimProduct src
ON trg.ProductID = src.ProductID
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
select * from sales_scd.DimProduct